# EEAI-Chile — Pilot Study: Internal-Consistency Reliability (APA7)

This notebook estimates the internal-consistency reliability of the instrument (6 dimensions + total scale, 29 items, 5-point Likert) and exports two APA 7th-edition tables to a single Word document:

- **Table 3** — Reliability by dimension and total: Cronbach's α, ordinal α, and McDonald's ω computed two ways for comparison — **continuous** (Pearson) and **ordinal** (polychoric) — each with an **analytic 95% CI** (delta method).
- **Table 4** — Item-level diagnostics: M, SD, corrected item–total correlation, and α-if-item-deleted.

Coefficients for ordinal data are computed from **polychoric** correlations. No factor model is imposed beyond a single-factor congeneric model used **only** to obtain ω per subscale (this is not an exploratory/confirmatory factor analysis of the full instrument). Run the cells in order in Google Colab.

In [1]:
# ==========================================================================
# Install dependencies (safe to re-run)
# ==========================================================================
!pip install -q python-docx openpyxl

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

## 1. Load data and define dimensions

In [2]:
# ==========================================================================
# Configuration, data loading, and dimension -> item mapping
# ==========================================================================
import numpy as np, pandas as pd
from IPython.display import display

DATA_DIR    = "../data/pilot"
XLSX_PATH   = f"{DATA_DIR}/EEAI-Chile_Pilot_Data_2026_anonymized.xlsx"
OUTPUT_DOCX = "../outputs/tables/EEAI_Pilot_Reliability_APA7.docx"

df = pd.read_excel(XLSX_PATH, sheet_name=0)
C  = df.columns

# The 29 instrument items are columns 16..44 (0-based), in dimension order.
# Each dimension: (code, short label [Table 3], full label [Table 4], column range)
DIMS = [
    ("D1", "D1. Vitality, dedication, and absorption",
           "D1. Teaching vitality, dedication, and absorption", range(16, 21)),
    ("D2", "D2. Social engagement and impact",
           "D2. Social engagement and impact",                  range(21, 26)),
    ("D3", "D3. Digital agency and IT",
           "D3. Digital agency and information technologies",   range(26, 30)),
    ("D4", "D4. Fairness and contractual environment",
           "D4. Fairness and contractual environment",          range(30, 35)),
    ("D5", "D5. Academic leadership and support",
           "D5. Academic leadership and support",               range(35, 40)),
    ("D6", "D6. Governance and strategic participation",
           "D6. Governance and strategic participation",        range(40, 45)),
]
ITEM_IDX = list(range(16, 45))  # all 29 items, in order

# Numeric item matrix and complete-case N
items_all = df[[C[i] for i in ITEM_IDX]].apply(pd.to_numeric, errors="coerce")
N = int(items_all.dropna().shape[0])
print(f"{len(ITEM_IDX)} items across {len(DIMS)} dimensions; complete-case N = {N}")
print("Response scale observed:", sorted(pd.unique(items_all.values.ravel())))

29 items across 6 dimensions; complete-case N = 104
Response scale observed: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]


## 2. Reliability functions (polychoric, α, ordinal α, ω, bootstrap)

In [3]:
# ==========================================================================
# Self-contained reliability toolkit (no external stats packages required)
# ==========================================================================
from scipy.stats import norm, f as _fdist
from scipy.optimize import minimize_scalar, minimize

# Gauss-Legendre nodes/weights for the Genz bivariate-normal CDF
_W = {1:(np.array([0.1713244923791705,0.3607615730481384,0.4679139345726904]),
         np.array([0.9324695142031522,0.6612093864662647,0.2386191860831970])),
      2:(np.array([0.04717533638651177,0.1069393259953183,0.1600783285433464,
                   0.2031674267230659,0.2334925365383547,0.2491470458134029]),
         np.array([0.9815606342467191,0.9041172563704750,0.7699026741943050,
                   0.5873179542866171,0.3678314989981802,0.1252334085114692])),
      3:(np.array([0.01761400713915212,0.04060142980038694,0.06267204833410906,
                   0.08327674157670475,0.1019301198172404,0.1181945319615184,
                   0.1316886384491766,0.1420961093183821,0.1491729864726037,
                   0.1527533871307259]),
         np.array([0.9931285991850949,0.9639719272779138,0.9122344282513259,
                   0.8391169718222188,0.7463319064601508,0.6360536807265150,
                   0.5108670019508271,0.3737060887154196,0.2277858511416451,
                   0.07652652113349733]))}
def _nw(r): return _W[1] if abs(r)<0.3 else (_W[2] if abs(r)<0.75 else _W[3])

def _bvnu_scalar(dh, dk, r):
    if np.isinf(dh) or np.isinf(dk): return 0.0
    if r==0: return norm.sf(dh)*norm.sf(dk)
    tp=2*np.pi; w,x=_nw(r); h,k=dh,dk; hk=h*k; bvn=0.0
    if abs(r)<0.925:
        hs=(h*h+k*k)/2; asr=np.arcsin(r)/2
        for wi,xi in zip(np.concatenate([w,w]), np.concatenate([-x,x])):
            sn=np.sin(asr*(xi+1)); bvn+=wi*np.exp((sn*hk-hs)/(1-sn*sn))
        bvn=bvn*asr/tp+norm.cdf(-h)*norm.cdf(-k)
    else:
        if r<0: k=-k; hk=-hk
        if abs(r)<1:
            as_=1-r*r; a=np.sqrt(as_); bs=(h-k)**2; asr=-(bs/as_+hk)/2
            c=(4-hk)/8; d=(12-hk)/80
            if asr>-100: bvn=a*np.exp(asr)*(1-c*(bs-as_)*(1-d*bs)/3+c*d*as_*as_)
            if -hk<100:
                b=np.sqrt(bs); sp=np.sqrt(tp)*norm.cdf(-b/a)
                bvn-=np.exp(-hk/2)*sp*b*(1-c*bs*(1-d*bs)/3)
            a=a/2
            for wi,xi in zip(np.concatenate([w,w]), np.concatenate([-x,x])):
                xs=(a*(xi+1))**2; rs=np.sqrt(1-xs); asr1=-(bs/xs+hk)/2
                if asr1>-100:
                    sp=1+c*xs*(1+d*xs); ep=np.exp(-hk*xs/(2*(1+rs)**2))/rs
                    bvn+=a*wi*np.exp(asr1)*(ep-sp)
            bvn=-bvn/tp
        if r>0: bvn+=norm.cdf(-max(h,k))
        else:
            bvn=-bvn
            if k>h: bvn+=norm.cdf(k)-norm.cdf(h)
    return min(max(bvn,0.0),1.0)

def _bvn_upper_grid(H,K,r):
    w,x=_nw(r); hk=H*K; hs=(H*H+K*K)/2; asr=np.arcsin(r)/2
    bvn=np.zeros_like(H,dtype=float)
    for wi,xi in zip(np.concatenate([w,w]), np.concatenate([-x,x])):
        sn=np.sin(asr*(xi+1)); bvn+=wi*np.exp((sn*hk-hs)/(1-sn*sn))
    return bvn*asr/(2*np.pi)+norm.cdf(-H)*norm.cdf(-K)

def _phi2_grid(a,b,r):
    A,B=np.meshgrid(a,b,indexing="ij"); out=np.empty_like(A,dtype=float)
    fin=np.isfinite(A)&np.isfinite(B); Hh,Kk=-A[fin],-B[fin]
    if abs(r)<0.925: out[fin]=_bvn_upper_grid(Hh,Kk,r)
    else: out[fin]=np.array([_bvnu_scalar(h,k,r) for h,k in zip(Hh,Kk)])
    out[np.isneginf(A)|np.isneginf(B)]=0.0
    mA=np.isposinf(A); out[mA]=norm.cdf(B[mA])
    mB=np.isposinf(B); out[mB]=norm.cdf(A[mB])
    out[np.isposinf(A)&np.isposinf(B)]=1.0
    return out

def _thr(counts):
    cum=np.cumsum(counts)/counts.sum(); cum=np.clip(cum[:-1],1e-6,1-1e-6)
    return np.concatenate([[-np.inf],norm.ppf(cum),[np.inf]])

def polychoric_corr(x,y):
    """Two-step (Olsson) polychoric correlation between two ordinal items."""
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=~(np.isnan(x)|np.isnan(y)); x,y=x[m],y[m]
    xs=sorted(np.unique(x)); ys=sorted(np.unique(y))
    xi={v:i for i,v in enumerate(xs)}; yi={v:i for i,v in enumerate(ys)}
    kx,ky=len(xs),len(ys)
    if kx<2 or ky<2: return np.nan
    tab=np.zeros((kx,ky))
    for vx,vy in zip(x,y): tab[xi[vx],yi[vy]]+=1
    a=_thr(tab.sum(1)); b=_thr(tab.sum(0))
    def negll(r):
        P=_phi2_grid(a,b,r); cell=P[1:,1:]-P[:-1,1:]-P[1:,:-1]+P[:-1,:-1]
        return -np.sum(tab*np.log(np.clip(cell,1e-12,1)))
    return minimize_scalar(negll,bounds=(-0.999,0.999),method="bounded",
                           options={"xatol":1e-3}).x

def polychoric_matrix(data):
    data=np.asarray(data,float); k=data.shape[1]; R=np.eye(k)
    for i in range(k):
        for j in range(i+1,k):
            R[i,j]=R[j,i]=polychoric_corr(data[:,i],data[:,j])
    return R

def cronbach_alpha(data):
    data=np.asarray(data,float); k=data.shape[1]
    iv=data.var(axis=0,ddof=1); tv=data.sum(axis=1).var(ddof=1)
    return k/(k-1)*(1-iv.sum()/tv)

def ordinal_alpha_from_R(R):
    k=R.shape[0]; rbar=(R.sum()-k)/(k*(k-1))
    return k*rbar/(1+(k-1)*rbar)

def _minres_1factor(R):
    """Single-factor MINRES loadings (minimize off-diagonal residuals of R - λλ')."""
    k=R.shape[0]; off=~np.eye(k,dtype=bool)
    ev,evec=np.linalg.eigh(R); lam0=evec[:,-1]*np.sqrt(max(ev[-1],1e-6))
    if lam0.sum()<0: lam0=-lam0
    res=minimize(lambda l:np.sum(((R-np.outer(l,l))[off])**2),lam0,
                 method="L-BFGS-B",bounds=[(-0.999,0.999)]*k)
    lam=res.x
    return -lam if lam.sum()<0 else lam

def omega_from_R(R):
    lam=_minres_1factor(R); s=lam.sum()
    return s*s/(s*s+np.sum(1-lam**2))

def mean_interitem_R(R):
    k=R.shape[0]; return (R.sum()-k)/(k*(k-1))

def corrected_item_total(data):
    data=np.asarray(data,float); k=data.shape[1]; out=[]
    for i in range(k):
        rest=np.delete(data,i,axis=1).sum(axis=1)
        out.append(np.corrcoef(data[:,i],rest)[0,1])
    return np.array(out)

def alpha_if_deleted(data):
    data=np.asarray(data,float); k=data.shape[1]
    return np.array([cronbach_alpha(np.delete(data,i,axis=1)) if k>2 else np.nan
                     for i in range(k)])

def coefficients(data):
    R=polychoric_matrix(data)
    return dict(alpha=cronbach_alpha(data), oalpha=ordinal_alpha_from_R(R),
                omega=omega_from_R(R), mean_r=mean_interitem_R(R))

def corr_acov(R, n):
    """Asymptotic covariance of the unique off-diagonal correlations under
    normality (Olkin-Siotani / Steiger, 1980). Returns (Acov, pairs)."""
    k=R.shape[0]; pairs=[(i,j) for i in range(k) for j in range(i+1,k)]; m=len(pairs)
    A=np.zeros((m,m))
    for p,(i,j) in enumerate(pairs):
        for q,(kk,mm) in enumerate(pairs):
            rij=R[i,j]; rkm=R[kk,mm]
            rik=R[i,kk]; rim=R[i,mm]; rjk=R[j,kk]; rjm=R[j,mm]
            A[p,q]=(0.5*rij*rkm*(rik**2+rim**2+rjk**2+rjm**2)
                    +rik*rjm+rim*rjk
                    -rij*(rik*rim+rjk*rjm)-rkm*(rik*rjk+rim*rjm))/n
    return A,pairs

def coef_ci_delta(R, n, fn=None, level=0.95, logit=True):
    """Delta-method 95% CI for a reliability coefficient that is a function fn(R)
    of a correlation matrix (used for ordinal alpha and for both omegas)."""
    if fn is None: fn = omega_from_R
    Acov, pairs = corr_acov(R, n); c0 = fn(R); eps = 1e-4
    J = np.zeros(len(pairs))
    for p,(i,j) in enumerate(pairs):
        Rp=R.copy(); Rp[i,j]+=eps; Rp[j,i]+=eps
        Rm=R.copy(); Rm[i,j]-=eps; Rm[j,i]-=eps
        J[p]=(fn(Rp)-fn(Rm))/(2*eps)
    se=np.sqrt(max(float(J@Acov@J),0.0)); z=norm.ppf(1-(1-level)/2)
    if logit and 0<c0<1:
        g=np.log(c0/(1-c0)); sg=se/(c0*(1-c0))
        lo=1/(1+np.exp(-(g-z*sg))); hi=1/(1+np.exp(-(g+z*sg)))
    else:
        lo,hi=c0-z*se, c0+z*se
    return c0, max(lo,0.0), min(hi,1.0)

def cronbach_alpha_ci(data, level=0.95):
    """Feldt (1965) F-based 95% CI for Cronbach's alpha (the interval JASP reports)."""
    data=np.asarray(data,float); n,k=data.shape; a=cronbach_alpha(data)
    g=1-level; df1=n-1; df2=(n-1)*(k-1)
    FL=_fdist.ppf(1-g/2, df1, df2); FU=_fdist.ppf(g/2, df1, df2)
    return a, 1-(1-a)*FL, 1-(1-a)*FU
print("Reliability toolkit ready.")

Reliability toolkit ready.


## 3. Compute Table 3 (reliability) and Table 4 (item diagnostics)

For each dimension and the total scale we compute Cronbach's α, ordinal α, and McDonald's ω in **two versions** — continuous (Pearson) and ordinal (polychoric) — each with an **analytic 95% CI** (delta method, Olkin–Siotani/Steiger normal-theory covariances). This runs in well under a minute (no bootstrap).

In [4]:
# ==========================================================================
# Compute Table 3 (reliability) and Table 4 (item diagnostics, by dimension)
# ==========================================================================
import time

def f_corr(x):
    """2 decimals, no leading zero (APA); handles sign and NaN."""
    if x is None or (isinstance(x, float) and np.isnan(x)): return "—"
    s = f"{x:.2f}"
    return s.replace("0.", ".", 1) if s.startswith(("0.", "-0.")) else s
def f_stat(x): return f"{x:.2f}"
def f_om(om, lo, hi): return f"{f_corr(om)} [{f_corr(lo)}, {f_corr(hi)}]"

t0 = time.time()
rows3, rows4 = [], []

def add_block(label_short, label_full, data, item_numbers):
    n = data.shape[0]
    Rp = np.corrcoef(data, rowvar=False)     # Pearson  -> continuous
    Ro = polychoric_matrix(data)             # polychoric -> ordinal
    a,  al,  ah  = cronbach_alpha_ci(data)            # Feldt CI
    oa, oal, oah = coef_ci_delta(Ro, n, ordinal_alpha_from_R)  # delta CI
    omc, loc, hic = coef_ci_delta(Rp, n, omega_from_R)
    omo, loo, hio = coef_ci_delta(Ro, n, omega_from_R)
    rows3.append([label_short, str(data.shape[1]),
                  f_om(a, al, ah), f_om(oa, oal, oah),
                  f_om(omc, loc, hic), f_om(omo, loo, hio)])
    # item rows (by dimension) — only when item_numbers is not None
    if item_numbers is not None:
        rows4.append([label_full, "", "", "", "", "", "", ""])
        kk = data.shape[1]
        lam = _minres_1factor(Ro)               # standardized loadings (ordinal)
        rit = corrected_item_total(data)        # corrected item-rest (Pearson)
        aid = alpha_if_deleted(data)            # Cronbach alpha if deleted
        oid = [omega_from_R(Ro[np.ix_([j for j in range(kk) if j!=p],
                                      [j for j in range(kk) if j!=p])]) if kk > 2 else np.nan
               for p in range(kk)]              # ordinal omega if deleted
        M = data.mean(axis=0); SD = data.std(axis=0, ddof=1)
        for p, itnum in enumerate(item_numbers):
            rows4.append(["", f"Item {itnum}", f_stat(M[p]), f_stat(SD[p]),
                          f_corr(lam[p]), f_corr(rit[p]), f_corr(aid[p]), f_corr(oid[p])])

for code_, short_lbl, full_lbl, rng_ in DIMS:
    data = df[[C[i] for i in rng_]].apply(pd.to_numeric, errors="coerce").dropna().values
    item_numbers = [list(ITEM_IDX).index(i) + 1 for i in rng_]
    add_block(short_lbl, full_lbl, data, item_numbers)
    print(f"  {code_} done ({time.time()-t0:.1f}s)")

# Total scale (reliability only; item diagnostics stay at the dimension level)
add_block("Total scale", None, items_all.dropna().values, None)
print(f"Done in {time.time()-t0:.1f}s.")

table3 = pd.DataFrame(rows3, columns=[
    "Dimension", "Items", "α", "Ordinal α",
    "ω (continuous)", "ω (ordinal)"])
table4 = pd.DataFrame(rows4, columns=[
    "Dimension", "Item", "M", "SD", "λ", "Item–total r",
    "α if deleted", "ω if deleted"])
display(table3)
display(table4)

  D1 done (0.2s)


  D2 done (0.3s)


  D3 done (0.4s)


  D4 done (0.5s)


  D5 done (0.7s)


  D6 done (0.9s)


Done in 16.8s.


,Dimension,Items,α,Ordinal α,ω (continuous),ω (ordinal)
0,"D1. Vitality, dedication, and absorption",5,".68 [.57, .77]",".82 [.76, .87]",".76 [.69, .82]",".84 [.80, .88]"
1,D2. Social engagement and impact,5,".77 [.69, .83]",".84 [.79, .88]",".78 [.71, .84]",".84 [.79, .89]"
2,D3. Digital agency and IT,4,".76 [.68, .83]",".83 [.76, .87]",".77 [.69, .83]",".83 [.76, .87]"
3,D4. Fairness and contractual environment,5,".81 [.74, .86]",".84 [.79, .88]",".82 [.76, .86]",".85 [.80, .89]"
4,D5. Academic leadership and support,5,".90 [.86, .93]",".93 [.91, .95]",".90 [.87, .93]",".93 [.91, .95]"
5,D6. Governance and strategic participation,5,".77 [.69, .83]",".85 [.80, .89]",".79 [.72, .84]",".86 [.81, .89]"
6,Total scale,29,".92 [.89, .94]",".94 [.92, .95]",".92 [.89, .94]",".94 [.92, .96]"


,Dimension,Item,M,SD,λ,Item–total r,α if deleted,ω if deleted
0,"D1. Teaching vitality, dedication, and absorption",,,,,,,
1,,Item 1,4.62,0.66,.93,.63,.56,.75
2,,Item 2,4.64,0.62,.89,.62,.57,.76
3,,Item 3,4.64,0.67,.77,.54,.59,.80
4,,Item 4,4.40,0.76,.53,.34,.67,.85
5,,Item 5,3.89,1.05,.38,.24,.76,.87
6,D2. Social engagement and impact,,,,,,,
7,,Item 6,4.02,0.90,.82,.63,.69,.79
8,,Item 7,4.33,0.78,.73,.54,.73,.81
9,,Item 8,4.21,0.96,.69,.54,.72,.82


## 4. APA7 formatting helpers

In [5]:
# ==========================================================================
# APA7 Word helpers (bold "Table N", italic title, horizontal rules only, italic Note.)
# ==========================================================================
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

APA_FONT="Times New Roman"; APA_BODY_SIZE=11; APA_TABLE_SIZE=10

def _set_cell_border(cell,**kw):
    tcPr=cell._tc.get_or_add_tcPr(); b=tcPr.find(qn("w:tcBorders"))
    if b is None: b=OxmlElement("w:tcBorders"); tcPr.append(b)
    for edge in ("top","bottom","left","right"):
        if edge in kw:
            el=b.find(qn(f"w:{edge}"))
            if el is None: el=OxmlElement(f"w:{edge}"); b.append(el)
            for k,v in kw[edge].items(): el.set(qn(f"w:{k}"),str(v))

def _no_borders(c):
    _set_cell_border(c,top={"val":"nil"},bottom={"val":"nil"},left={"val":"nil"},right={"val":"nil"})

def _style_run(run,bold=False,italic=False,size=APA_BODY_SIZE,font=APA_FONT):
    run.bold=bold; run.italic=italic; run.font.size=Pt(size); run.font.name=font
    rPr=run._element.get_or_add_rPr(); rF=rPr.find(qn("w:rFonts"))
    if rF is None: rF=OxmlElement("w:rFonts"); rPr.append(rF)
    for a in ("w:ascii","w:hAnsi","w:cs"): rF.set(qn(a),font)

def _add_paragraph(doc,segs,align=WD_ALIGN_PARAGRAPH.LEFT,space_before=0,space_after=0,line_spacing=1.0):
    p=doc.add_paragraph(); p.alignment=align; pf=p.paragraph_format
    pf.space_before=Pt(space_before); pf.space_after=Pt(space_after); pf.line_spacing=line_spacing
    for t,s in segs: _style_run(p.add_run(t),**s)
    return p

def _set_cell_text(cell,text,bold=False,italic=False,align=WD_ALIGN_PARAGRAPH.LEFT,size=APA_TABLE_SIZE):
    cell.text=""; p=cell.paragraphs[0]; p.alignment=align
    p.paragraph_format.space_before=Pt(1); p.paragraph_format.space_after=Pt(1); p.paragraph_format.line_spacing=1.0
    _style_run(p.add_run("" if text is None else str(text)),bold=bold,italic=italic,size=size)

def _set_col_widths(table,widths):
    table.autofit=False; table.allow_autofit=False
    tbl=table._tbl; tblPr=tbl.tblPr; lay=tblPr.find(qn("w:tblLayout"))
    if lay is None: lay=OxmlElement("w:tblLayout"); tblPr.append(lay)
    lay.set(qn("w:type"),"fixed")
    grid=tbl.find(qn("w:tblGrid"))
    if grid is not None:
        for gc in list(grid.findall(qn("w:gridCol"))): grid.remove(gc)
        for w in widths:
            gc=OxmlElement("w:gridCol"); gc.set(qn("w:w"),str(int(w.inches*1440))); grid.append(gc)
    for row in table.rows:
        for j,w in enumerate(widths): row.cells[j].width=w

def add_apa_table(doc,df_in,table_number,title,note=None,bold_section_col=None,
                  col_aligns=None,table_font_size=APA_TABLE_SIZE,italic_headers=None,col_widths=None):
    _add_paragraph(doc,[(f"Table {table_number}",dict(bold=True))],space_before=12,space_after=0)
    _add_paragraph(doc,[(title,dict(italic=True))],space_after=4)
    cols=list(df_in.columns); n_rows=len(df_in)+1; n_cols=len(cols)
    italic_headers=set(italic_headers or []); col_aligns=col_aligns or {}
    table=doc.add_table(rows=n_rows,cols=n_cols); table.alignment=WD_TABLE_ALIGNMENT.CENTER
    for j,col in enumerate(cols):
        c=table.cell(0,j); _no_borders(c)
        _set_cell_text(c,col,bold=True,italic=(col in italic_headers),
                       align=(WD_ALIGN_PARAGRAPH.CENTER if j>0 else WD_ALIGN_PARAGRAPH.LEFT),size=table_font_size)
    for i in range(len(df_in)):
        row=df_in.iloc[i]; is_sec=False
        if bold_section_col is not None:
            v=row[bold_section_col]
            is_sec=(isinstance(v,str) and v.strip()!="" and
                    all(str(row[c]).strip()=="" for c in cols if c!=bold_section_col))
        for j,col in enumerate(cols):
            c=table.cell(i+1,j); _no_borders(c)
            al=col_aligns.get(col,WD_ALIGN_PARAGRAPH.LEFT if j==0 else WD_ALIGN_PARAGRAPH.CENTER)
            _set_cell_text(c,row[col],bold=is_sec,align=al,size=table_font_size)
    line={"val":"single","sz":6,"color":"000000","space":0}
    for j in range(n_cols):
        _set_cell_border(table.cell(0,j),top=line,bottom=line)
        _set_cell_border(table.cell(n_rows-1,j),bottom=line)
    if col_widths is not None: _set_col_widths(table,col_widths)
    if note: _add_paragraph(doc,[("Note. ",dict(italic=True)),(note,dict())],space_before=4,space_after=6)
    else: doc.add_paragraph()
print("APA helpers ready.")

APA helpers ready.


## 5. Export both tables to Word (APA7) and download

In [6]:
# ==========================================================================
# Assemble the Word document
# ==========================================================================
import os

TABLE3_NUMBER = 8   # reliability summary
TABLE4_NUMBER = 10   # item-level diagnostics

doc = Document()
nm = doc.styles["Normal"]; nm.font.name = APA_FONT; nm.font.size = Pt(APA_BODY_SIZE)

add_apa_table(
    doc, table3, TABLE3_NUMBER,
    "Internal-Consistency Reliability by Dimension and Total Scale",
    note=(f"N = {N}. Bracketed values are 95% confidence intervals. CIs for α use the Feldt "
          "(1965) F-based method; CIs for ordinal α, ω (continuous), and ω (ordinal) use the "
          "delta method (Olkin–Siotani/Steiger normal-theory covariances). α = Cronbach’s "
          "alpha; Ordinal α = standardized alpha on the polychoric correlation matrix. "
          "ω (continuous) = McDonald’s omega from a single-factor model on the Pearson "
          "covariance matrix; ω (ordinal) = the same on the polychoric matrix. α and "
          "ω (continuous) treat the responses as interval; ordinal α and ω (ordinal) treat "
          "them as ordinal (recommended for Likert items). Coefficients are reported without a "
          f"leading zero. Corrected item–total statistics are reported by dimension in Table {TABLE4_NUMBER}."),
    table_font_size=9,
    col_widths=[Inches(1.45), Inches(0.45), Inches(1.13), Inches(1.13),
                Inches(1.17), Inches(1.17)],
)

add_apa_table(
    doc, table4, TABLE4_NUMBER,
    "Item-Level Descriptive Statistics and Discrimination Indices",
    note=(f"N = {N}. M and SD are computed on the 1–5 response scale. λ = standardized "
          "loading on the single-factor model fitted to the polychoric correlation matrix "
          "(ordinal). Item–total r = corrected item–total (item-rest) correlation. "
          "α if deleted = Cronbach’s alpha of the dimension after removing the item; "
          "ω if deleted = ordinal McDonald’s omega of the dimension after removing the item. "
          "All item statistics are computed within each item’s dimension. Correlations and "
          "reliability coefficients are reported without a leading zero."),
    bold_section_col="Dimension",
    italic_headers=["M", "SD", "Item–total r"],
    table_font_size=9,
    col_widths=[Inches(2.0), Inches(0.6), Inches(0.42), Inches(0.42), Inches(0.5),
                Inches(0.82), Inches(0.86), Inches(0.86)],
)

doc.save(OUTPUT_DOCX)
print(f"✅ Saved: {OUTPUT_DOCX}")


✅ Saved: ../outputs/tables/EEAI_Pilot_Reliability_APA7.docx
